[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module3/06-pdf-excel.ipynb)

# Module 3.6 — PDF and Excel Automation
**Module 3: Automation & Scripting** | Estimated time: 25 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Create and style Excel workbooks with `openpyxl`
- Read and write Excel files with `pandas`
- Extract text from PDF files with `pypdf`
- Merge multiple PDFs into one
- Generate professional PDFs (with tables and styled text) using `reportlab`
- Build a complete invoice-style PDF report

In [ ]:
!pip install openpyxl pypdf reportlab pandas -q

import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, Reference
from openpyxl.utils import get_column_letter
import pandas as pd
from pypdf import PdfReader, PdfWriter
from reportlab.lib.pagesizes import A4, letter
from reportlab.lib import colors
from reportlab.lib.units import inch, cm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Table, TableStyle, Spacer, HRFlowable
from reportlab.platypus import KeepTogether
from io import BytesIO
from pathlib import Path
import datetime

OUTDIR = Path('/tmp/pypath_docs')
OUTDIR.mkdir(exist_ok=True)
print('openpyxl:', openpyxl.__version__)
print('pandas  :', pd.__version__)
print('Output dir:', OUTDIR)

## 1. Creating an Excel Workbook with `openpyxl`

In [ ]:
wb = Workbook()
ws = wb.active
ws.title = 'Sales Q1'

# ── Styles ───────────────────────────────────────────────────────────────
header_font   = Font(bold=True, color='FFFFFF', size=12)
header_fill   = PatternFill('solid', fgColor='2F75B6')
center_align  = Alignment(horizontal='center', vertical='center')
thin          = Side(style='thin', color='000000')
border        = Border(left=thin, right=thin, top=thin, bottom=thin)
alt_fill      = PatternFill('solid', fgColor='DDEEFF')
total_font    = Font(bold=True, size=11)

# ── Headers ───────────────────────────────────────────────────────────────
headers = ['Month', 'Product', 'Units Sold', 'Unit Price', 'Revenue']
for col, header in enumerate(headers, start=1):
    cell = ws.cell(row=1, column=col, value=header)
    cell.font       = header_font
    cell.fill       = header_fill
    cell.alignment  = center_align
    cell.border     = border

# ── Data ─────────────────────────────────────────────────────────────────
data = [
    ('Jan', 'Widget A', 120, 29.99),
    ('Jan', 'Widget B',  85, 49.99),
    ('Feb', 'Widget A', 145, 29.99),
    ('Feb', 'Widget B',  92, 49.99),
    ('Mar', 'Widget A', 178, 29.99),
    ('Mar', 'Widget B', 110, 49.99),
]
for row_idx, (month, product, units, price) in enumerate(data, start=2):
    fill = alt_fill if row_idx % 2 == 0 else PatternFill()
    for col, value in enumerate([month, product, units, price, units * price], start=1):
        cell = ws.cell(row=row_idx, column=col, value=value)
        cell.border = border
        if fill.fill_type:
            cell.fill = fill
        if col in (4, 5):
            cell.number_format = '"$"#,##0.00'

# ── Totals row ────────────────────────────────────────────────────────────
total_row = len(data) + 2
ws.cell(row=total_row, column=1, value='TOTAL').font = total_font
ws.cell(row=total_row, column=3, value=f'=SUM(C2:C{total_row-1})').font = total_font
ws.cell(row=total_row, column=5, value=f'=SUM(E2:E{total_row-1})').font = total_font
ws.cell(row=total_row, column=5).number_format = '"$"#,##0.00'

# ── Column widths ─────────────────────────────────────────────────────────
for col, width in zip('ABCDE', [10, 12, 12, 12, 14]):
    ws.column_dimensions[col].width = width

ws.row_dimensions[1].height = 22

path = OUTDIR / 'sales_q1.xlsx'
wb.save(path)
print(f'Saved: {path}  ({path.stat().st_size:,} bytes)')

## 2. Adding a Chart to Excel

In [ ]:
# Add a bar chart showing revenue per row
chart = BarChart()
chart.type         = 'col'
chart.title        = 'Revenue by Month & Product'
chart.y_axis.title = 'Revenue ($)'
chart.x_axis.title = 'Month'
chart.shape        = 4

data_ref  = Reference(ws, min_col=5, min_row=1, max_row=len(data) + 1)
labels    = Reference(ws, min_col=1, min_row=2, max_row=len(data) + 1)
chart.add_data(data_ref, titles_from_data=True)
chart.set_categories(labels)
chart.width  = 18
chart.height = 12

ws.add_chart(chart, 'G2')
wb.save(path)
print('Chart added and workbook saved.')

## 3. Reading and Writing Excel with `pandas`

In [ ]:
# Read the Excel file we just created
df = pd.read_excel(path, sheet_name='Sales Q1')
print('Read from Excel:')
print(df.head())
print(f'Shape: {df.shape}\n')

# Create a summary dataframe
summary = df.groupby('Month').agg(
    total_units=('Units Sold', 'sum'),
    total_revenue=('Revenue', 'sum')
).reset_index()
print('Monthly Summary:')
print(summary)

# Write multiple sheets at once
summary_path = OUTDIR / 'sales_summary.xlsx'
with pd.ExcelWriter(summary_path, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Raw Data', index=False)
    summary.to_excel(writer, sheet_name='Summary', index=False)

print(f'\nSaved multi-sheet workbook: {summary_path}')

## 4. Extracting Text from PDFs with `pypdf`

In [ ]:
# First, create a simple PDF to read back using reportlab
from reportlab.pdfgen import canvas as rl_canvas

sample_pdf_path = OUTDIR / 'sample.pdf'
c = rl_canvas.Canvas(str(sample_pdf_path), pagesize=A4)
w, h = A4

c.setFont('Helvetica-Bold', 18)
c.drawString(72, h - 72, 'PyPath Module 3 — Sample PDF')
c.setFont('Helvetica', 12)
c.drawString(72, h - 110, 'This document was generated by reportlab for demonstration purposes.')
c.drawString(72, h - 130, 'Page 1 of 2')
c.showPage()

c.setFont('Helvetica-Bold', 14)
c.drawString(72, h - 72, 'Page 2 Content')
c.setFont('Helvetica', 12)
for i, line in enumerate([
    'Python automation is powerful.',
    'PDF parsing with pypdf is straightforward.',
    'You can extract text from any page.',
], start=1):
    c.drawString(72, h - 72 - i * 20, f'{i}. {line}')
c.showPage()
c.save()
print(f'Sample PDF created: {sample_pdf_path}')

# Now read it back
reader = PdfReader(str(sample_pdf_path))
print(f'\nPDF has {len(reader.pages)} pages')
print(f'Encrypted: {reader.is_encrypted}')
print()
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    print(f'--- Page {i+1} ---')
    print(text.strip())
    print()

## 5. Merging PDFs with `pypdf`

In [ ]:
# Create a second PDF
second_pdf = OUTDIR / 'second.pdf'
c2 = rl_canvas.Canvas(str(second_pdf), pagesize=A4)
w, h = A4
c2.setFont('Helvetica-Bold', 16)
c2.drawString(72, h - 72, 'Appendix A — Data Tables')
c2.setFont('Helvetica', 11)
c2.drawString(72, h - 100, 'This page was merged from a separate PDF document.')
c2.showPage()
c2.save()

# Merge both PDFs
writer = PdfWriter()
for pdf_path in [sample_pdf_path, second_pdf]:
    reader = PdfReader(str(pdf_path))
    for page in reader.pages:
        writer.add_page(page)

merged_path = OUTDIR / 'merged.pdf'
with open(merged_path, 'wb') as f:
    writer.write(f)

merged_reader = PdfReader(str(merged_path))
print(f'Merged PDF: {merged_path}')
print(f'Total pages: {len(merged_reader.pages)}')
print(f'File size  : {merged_path.stat().st_size:,} bytes')

## 6. Generating a Professional Invoice PDF with `reportlab`

In [ ]:
def generate_invoice(invoice_data: dict, output_path: str):
    """Generate a styled invoice PDF using reportlab Platypus."""
    doc = SimpleDocTemplate(
        output_path, pagesize=A4,
        rightMargin=2*cm, leftMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm
    )
    styles = getSampleStyleSheet()
    story  = []

    # ── Company header ────────────────────────────────────────────────────
    co_style = ParagraphStyle('co', fontSize=22, textColor=colors.HexColor('#2F75B6'),
                              fontName='Helvetica-Bold', spaceAfter=4)
    story.append(Paragraph(invoice_data['company'], co_style))
    story.append(Paragraph(invoice_data['address'], styles['Normal']))
    story.append(Spacer(1, 0.3*cm))
    story.append(HRFlowable(width='100%', thickness=2, color=colors.HexColor('#2F75B6')))
    story.append(Spacer(1, 0.5*cm))

    # ── Invoice meta ──────────────────────────────────────────────────────
    inv_style = ParagraphStyle('inv', fontSize=14, fontName='Helvetica-Bold',
                               textColor=colors.HexColor('#444444'))
    story.append(Paragraph(f"INVOICE #{invoice_data['invoice_no']}", inv_style))
    story.append(Paragraph(f"Date: {invoice_data['date']}", styles['Normal']))
    story.append(Paragraph(f"Due:  {invoice_data['due_date']}", styles['Normal']))
    story.append(Spacer(1, 0.5*cm))

    # ── Bill to ───────────────────────────────────────────────────────────
    story.append(Paragraph('<b>Bill To:</b>', styles['Normal']))
    story.append(Paragraph(invoice_data['client_name'], styles['Normal']))
    story.append(Paragraph(invoice_data['client_email'], styles['Normal']))
    story.append(Spacer(1, 0.7*cm))

    # ── Line items table ──────────────────────────────────────────────────
    table_data = [['Description', 'Qty', 'Unit Price', 'Total']]
    subtotal = 0
    for item in invoice_data['items']:
        total = item['qty'] * item['price']
        subtotal += total
        table_data.append([
            item['description'], str(item['qty']),
            f"${item['price']:.2f}", f"${total:.2f}"
        ])
    tax = subtotal * invoice_data.get('tax_rate', 0.1)
    grand_total = subtotal + tax
    table_data.append(['', '', 'Subtotal', f'${subtotal:.2f}'])
    table_data.append(['', '', f'Tax ({invoice_data.get("tax_rate", 0.1)*100:.0f}%)', f'${tax:.2f}'])
    table_data.append(['', '', 'TOTAL DUE', f'${grand_total:.2f}'])

    col_widths = [9*cm, 2*cm, 3*cm, 3*cm]
    t = Table(table_data, colWidths=col_widths)
    t.setStyle(TableStyle([
        ('BACKGROUND',  (0, 0), (-1, 0), colors.HexColor('#2F75B6')),
        ('TEXTCOLOR',   (0, 0), (-1, 0), colors.white),
        ('FONTNAME',    (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('FONTSIZE',    (0, 0), (-1, 0), 11),
        ('ALIGN',       (1, 0), (-1, -1), 'RIGHT'),
        ('ALIGN',       (0, 0), (0, -1), 'LEFT'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -4), [colors.white, colors.HexColor('#EEF4FB')]),
        ('LINEBELOW',   (0, 0), (-1, 0), 1, colors.HexColor('#2F75B6')),
        ('LINEABOVE',   (0, -3), (-1, -3), 1, colors.HexColor('#CCCCCC')),
        ('FONTNAME',    (2, -1), (-1, -1), 'Helvetica-Bold'),
        ('FONTSIZE',    (2, -1), (-1, -1), 12),
        ('TEXTCOLOR',   (2, -1), (-1, -1), colors.HexColor('#2F75B6')),
        ('TOPPADDING',  (0, 0), (-1, -1), 6),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 6),
    ]))
    story.append(t)
    story.append(Spacer(1, 1*cm))

    # ── Payment note ─────────────────────────────────────────────────────
    note_style = ParagraphStyle('note', fontSize=9, textColor=colors.grey, italic=1)
    story.append(Paragraph(
        f'Payment due by {invoice_data["due_date"]}. Thank you for your business!',
        note_style
    ))

    doc.build(story)


# Build the invoice
invoice = {
    'company'     : 'PyPath Solutions Ltd.',
    'address'     : '123 Python Ave, Tech City, CA 94000  |  billing@pypath.dev',
    'invoice_no'  : 'INV-2024-0042',
    'date'        : '2024-03-15',
    'due_date'    : '2024-04-15',
    'client_name' : 'Acme Corporation',
    'client_email': 'accounts@acme.example.com',
    'tax_rate'    : 0.08,
    'items': [
        {'description': 'Python Automation Consulting (10 hrs)', 'qty': 10,  'price': 150.00},
        {'description': 'Web Scraping Module Development',        'qty':  1,  'price': 800.00},
        {'description': 'API Integration & Testing',              'qty':  5,  'price': 120.00},
        {'description': 'Documentation & Training Materials',     'qty':  1,  'price': 350.00},
    ],
}

invoice_path = str(OUTDIR / 'invoice_INV-2024-0042.pdf')
generate_invoice(invoice, invoice_path)
size = Path(invoice_path).stat().st_size
print(f'Invoice generated: {invoice_path}')
print(f'File size: {size:,} bytes')

## Practice Exercises

**Exercise 1 — Conditional Formatting**  
Using `openpyxl`, open the `sales_q1.xlsx` file and add conditional formatting so that any Revenue cell below $3,000 is highlighted red and any Revenue cell above $5,000 is highlighted green. Save the result as `sales_q1_formatted.xlsx`.

**Exercise 2 — PDF Table of Contents**  
Using `pypdf`, open `merged.pdf` and print a "table of contents" listing the page number and the first non-empty line of text from each page. Format the output as a numbered list.

**Exercise 3 — pandas Excel Report**  
Create a DataFrame with 30 rows of fake employee data (`name`, `department`, `salary`, `start_date`). Use `pd.ExcelWriter` with `openpyxl` to write:
- Sheet 1: All employees, sorted by salary descending
- Sheet 2: Department summary with average salary and headcount
- Apply `set_column` widths to make the file readable without manual adjustment